# Module 302 — Clean Label Attack Study

HTB Academy, AI Data Attacks, Section 9: **Clean Label Attacks**. This notebook introduces the three-class manufacturing example, creates a reproducible synthetic dataset, makes a stratified train/test split, standardizes the features without test-set leakage, and plots the clean training data.

This section introduces the attack idea and prepares its dataset. It does **not** implement feature poisoning yet; that belongs to later course material. The labels in this notebook remain clean. The two synthetic measurements stand in for component length and weight; they are not real factory data.

## Read the notation aloud

- `x_i = (x_i1, x_i2)`: **“ex sub i equals the pair ex sub i one, ex sub i two.”** This is row `i` with its two feature measurements.
- `y_i`: **“why sub i.”** This is the ground-truth class label for row `i`.
- Conceptual feature edit: `x_i′ = x_i + δ_i`, **“ex sub i prime equals ex sub i plus delta sub i.”** `x_i′` is the altered feature row and `δ_i` is the vector of small changes. Then `δ_i = x_i′ − x_i`, **“delta sub i equals ex sub i prime minus ex sub i.”**
- Clean-label condition: `y_i′ = y_i`, **“why sub i prime equals why sub i.”** The feature row changes while its assigned label stays the same. This notation is a teaching shorthand for the attack idea; no feature edits are applied in this section.


## 1. Imports and repeatable settings

NumPy and scikit-learn create the toy dataset and split it. `StandardScaler` learns the feature means and standard deviations from training data. Matplotlib draws the three classes. The class-to-name mapping follows HTB's quality-control story: 0 is Major Defect, 1 is Acceptable, and 2 is Minor Defect.

This subsection only needs data setup and visualization. Attack-specific helpers are left for the sections that introduce them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 1337
CLASS_NAMES = {
    0: "Major Defect",
    1: "Acceptable",
    2: "Minor Defect",
}
CLASS_COLORS = {
    0: "#0086ff",  # blue
    1: "#ffaf00",  # orange
    2: "#ff3e3e",  # red
}


## 2. Generate the three-class feature matrix

`make_blobs` creates 1,500 points around three centers. Each row of `X_raw` is one two-measurement example, and the matching entry in `y` is its class number. The centers are separated enough that a later attack's effect should be visible in a plot.

For instance, the first feature might represent length and the second weight. Those names are only an analogy: the generated values are artificial.


In [ ]:
N_SAMPLES = 1_500
CENTERS = [(0, 6), (4, 3), (8, 6)]
CLUSTER_STD = 1.15

X_raw, y = make_blobs(
    n_samples=N_SAMPLES,
    centers=CENTERS,
    n_features=2,
    cluster_std=CLUSTER_STD,
    random_state=SEED,
)

print(f"Generated features: {X_raw.shape}  (rows, two measurements)")
print(f"Generated labels:   {y.shape}")
print(f"Class counts:       {np.bincount(y)}")
print(f"Class names:        {CLASS_NAMES}")


## 3. Split first and preserve class proportions

A 30% test fraction reserves 450 of the 1,500 examples for evaluation. `stratify=y` asks the split to keep each class's proportion similar in both partitions. With this balanced toy dataset, the expected split has 350 training and 150 test examples per class.

HTB's displayed setup standardizes before splitting. This study notebook keeps the same learning objective but splits first, because test examples should not influence training-time preprocessing.


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

print(f"Training rows: {len(y_train)}; class counts: {np.bincount(y_train)}")
print(f"Test rows:     {len(y_test)}; class counts: {np.bincount(y_test)}")
assert len(y_train) == 1_050
assert len(y_test) == 450
assert np.array_equal(np.bincount(y_train), [350, 350, 350])
assert np.array_equal(np.bincount(y_test), [150, 150, 150])


## 4. Standardize using training data only

For feature `k`, the standardization formula is

`z_ik = (x_ik − μ_k) / σ_k`.

Read it as **“z sub i k equals x sub i k minus mu sub k, divided by sigma sub k.”** `x_ik` is feature `k` of example `i`; `μ_k` is the training-set mean for that feature; `σ_k` is its training-set standard deviation. The result is measured in standard deviations from the mean. For example, if a measurement is 10, its training mean is 8, and its standard deviation is 2, the scaled value is `(10 − 8) / 2 = 1`.

The scaler first **fits** by calculating `μ_k` and `σ_k` from `X_train_raw`, then **transforms** both splits with those same values. `fit_transform` combines those two actions for training data; `transform` reuses the learned values for test data. If the test set helped choose the means and standard deviations, information from evaluation data would leak into training preparation.


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("Training feature means (approximately zero):", X_train.mean(axis=0))
print("Training feature standard deviations (approximately one):", X_train.std(axis=0))
print("Test features transformed with training-only scaler:", X_test.shape)
assert X_train.shape == (1_050, 2)
assert X_test.shape == (450, 2)
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(X_train.std(axis=0), 1.0, atol=1e-12)


## 5. Visualize the clean training examples

Each plotted dot is a training example in the standardized two-feature space. Color indicates its unchanged class. The legend translates the numeric labels back into the manufacturing story. This is a baseline picture before clean-label poisoning; there are no altered features in this section.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for class_id, class_name in CLASS_NAMES.items():
    rows = y_train == class_id
    ax.scatter(
        X_train[rows, 0],
        X_train[rows, 1],
        color=CLASS_COLORS[class_id],
        edgecolors="#141d2b",
        alpha=0.75,
        s=38,
        label=f"Class {class_id}: {class_name}",
    )

ax.set_title("Clean training data — before any feature poisoning")
ax.set_xlabel("Standardized measurement 1")
ax.set_ylabel("Standardized measurement 2")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()


## COAE quick recall — clean-label attack setup

- **Label flipping:** change `y_i`, leave feature row `x_i` fixed.
- **Clean-label poisoning:** subtly change feature row `x_i`, leave label `y_i` unchanged.
- **Goal:** often make a particular inference-time target receive a chosen wrong class.
- **“Clean” means:** the training labels remain plausible and unmodified; feature values may be poisoned.
- **Standardization:** subtract a feature's training mean and divide by its training standard deviation.
- **Leakage-safe preprocessing:** split first, fit the scaler on training features only, transform test features with that fitted scaler.
- **Stratification:** keeps approximate class proportions in train and test partitions.
- **This subsection's scope:** build and inspect the clean three-class dataset. Do not claim the attack has been implemented or succeeded yet.

**Exam check:** If a malicious training example is moved in feature space but keeps its original class label, that is clean-label poisoning. If the class label itself is changed, it is label flipping. Always state whether the attacker altered features, labels, or both.
